In [64]:
import sys 
sys.path.append("../")

import pandas as pd 
import plotly.graph_objects as go 
from dateutil import parser
import datetime as dt 

from api.oanda_api import OandaApi 
from infrastructure.instrument_collection import instrumentCollection as ic 
from db.db import DataDB

pd.set_option('display.max_rows', None)


In [2]:
database = DataDB()

In [3]:
ic.load_instrumentsDB()

In [69]:
pairs = []

for pair, val in ic.instruments_dict.items():
    if val.ins_type == "CURRENCY" and "USD" in pair:
        pairs.append(pair)

In [70]:
data = []
api = OandaApi()
for p in pairs:
    d_temp  = api.get_candles_df(pair_name=p, granularity="D", count=400)
    d_temp["gain"] = (d_temp.mid_c - d_temp.mid_o )* 100
    d_temp["pair"] = p
    if "_GBP" in p:
        d_temp["gain"] = d_temp["gain"] *  -1
    data.append(d_temp)

candles_df = pd.concat(data)
candles_df.reset_index(drop=True, inplace=True)
candles_df.time = candles_df.time.dt.date

In [71]:
candles_df.tail()

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,gain,pair
7995,2026-07-19,221054,17.52671,17.55336,17.40838,17.42905,17.49700,17.54915,17.40626,17.42520,17.55642,17.56504,17.41040,17.43290,-9.766,USD_MXN
7996,2026-07-20,174878,17.41376,17.43742,17.37585,17.40120,17.39460,17.42991,17.37330,17.39870,17.43293,17.45712,17.37830,17.40370,-1.256,USD_MXN
7997,2026-07-21,190020,17.42118,17.43586,17.38642,17.39618,17.41039,17.43371,17.38390,17.39125,17.43196,17.43800,17.38877,17.40110,-2.500,USD_MXN
7998,2026-07-22,232061,17.40366,17.53978,17.37710,17.52240,17.37903,17.53790,17.37460,17.51630,17.42830,17.54191,17.37910,17.52849,11.874,USD_MXN
7999,2026-07-23,181505,17.50931,17.52344,17.43958,17.48330,17.50050,17.51990,17.43749,17.47960,17.51812,17.53110,17.44149,17.48700,-2.601,USD_MXN


In [72]:
calendar_data = database.query_all(DataDB.CALANDER_COLL)

In [76]:
calendar_data_df = pd.DataFrame.from_dict(calendar_data)

In [77]:
calendar_data_df.head()

,date,country,category,event,symbol,actual,previous,forecast
0,2026-07-20,canada,inflation rate,inflation rate yoy,CACPIYOY,,3.2%,3%
1,2026-07-21,united kingdom,unemployment rate,unemployment rate,UKUEILOR,,4.9%,4.9%
2,2026-07-21,germany,zew economic sentiment index,zew economic sentiment index,GERMANYZEWECOSENIND,,10.5,15
3,2026-07-21,japan,balance of trade,balance of trade,JNTBAL,,¥-378.7B,¥ -700B
4,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,,2.8%,2.6%


In [78]:
calendar_data_df.date = calendar_data_df.date.dt.date

In [79]:
calendar_data_df_uk = calendar_data_df[calendar_data_df.country == "united states"].copy()

In [80]:
calendar_data_df_uk.head(10)

,date,country,category,event,symbol,actual,previous,forecast
15,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
31,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
47,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
63,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
79,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
95,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
111,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
127,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
143,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%
159,2026-07-27,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,,-4.5%,0.3%


In [81]:
pd.set_option('future.no_silent_downcasting', True)

for col in ['actual', "previous", "forecast"]:
    calendar_data_df_uk[col] = calendar_data_df_uk[col].astype(str)
    for sy in ["£", "%", "B", "k"]:
        calendar_data_df_uk[col] = calendar_data_df_uk[col].str.replace(sy, "", regex=False)
    calendar_data_df_uk[col].replace("", 0, inplace=True)
    calendar_data_df_uk[col] = calendar_data_df_uk[col].astype(float)

C:\Users\otavi\AppData\Local\Temp\ipykernel_15668\991631731.py:7: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



C:\Users\otavi\AppData\Local\Temp\ipykernel_15668\991631731.py:7: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

In [82]:
date_match = {}
for orig in calendar_data_df_uk.date.unique():
    d = orig 
    tries = 0 
    while d not in candles_df.time.values:
        d = d + dt.timedelta(days=1)
        tries += 1 
        if tries > 20:
            print("Failed")
            break
    date_match[orig] = d 


Failed


In [83]:
candle_times = set(candles_df.time.values)
date_match = {}
for orig in calendar_data_df_uk.date.unique():
    d = orig
    tries = 0
    while d not in candle_times:
        d = d + dt.timedelta(days=1)
        tries += 1
        if tries > 10:
            print(f"Failed for {orig}")
            break
    date_match[orig] = d

Failed for 2026-07-27


In [84]:
print("Calendar date range:", calendar_data_df_uk.date.min(), "to", calendar_data_df_uk.date.max())
print("Candles date range:", candles_df.time.min(), "to", candles_df.time.max())

Calendar date range: 2026-07-27 to 2026-07-27
Candles date range: 2025-01-08 to 2026-07-23


In [85]:
candle_times = set(candles_df.time.values)
date_match = {}
for orig in calendar_data_df_uk.date.unique():
    d = orig
    tries = 0
    matched = False
    while d not in candle_times:
        d = d + dt.timedelta(days=1)
        tries += 1
        if tries > 10:
            print(f"Failed for {orig}")
            break
    else:
        matched = True
    date_match[orig] = d if matched else pd.NaT

Failed for 2026-07-27


In [86]:
calendar_data_df_uk['orig_date'] = calendar_data_df_uk.date
calendar_data_df_uk.date = [date_match[x] for x in calendar_data_df_uk.date]

In [87]:
calendar_data_df_uk['delta_prev'] = calendar_data_df_uk.actual - calendar_data_df_uk.previous
calendar_data_df_uk['delta_fc'] = calendar_data_df_uk.actual - calendar_data_df_uk.forecast

In [88]:
calendar_data_df_uk.head(2)

,date,country,category,event,symbol,actual,previous,forecast,orig_date,delta_prev,delta_fc
15,NaT,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,0.0,-4.5,0.3,2026-07-27,4.5,-0.3
31,NaT,united states,durable goods orders,durable goods orders mom,UNITEDSTADURGOOORD,0.0,-4.5,0.3,2026-07-27,4.5,-0.3


In [89]:
candles_an.head(2)

,time,pair,gain
0,2025-01-08,GBP_AUD,-0.223
1,2025-01-09,GBP_AUD,0.034


In [90]:
candles_an = candles_df[['time', 'pair', 'gain']].copy()

In [91]:
merged = pd.merge(left=candles_an, right=calendar_data_df_uk, left_on="time", right_on="date")

ValueError: You are trying to merge on object and datetime64[ns] columns for key 'time'. If you wish to proceed you should use pd.concat

In [92]:
merged.category.unique()
merged[merged.category=="inflation rate"]

,time,pair,gain,date,country,category,event,symbol,actual,previous,forecast,orig_date,delta_prev,delta_fc
17,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
18,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
19,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
20,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
21,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
22,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
23,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
24,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
25,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6
26,2026-07-22,GBP_AUD,-0.149,2026-07-22,united kingdom,inflation rate,inflation rate yoy,UKRPCJYR,0.0,2.8,2.6,2026-07-22,-2.8,-2.6


In [93]:
c = "inflation rate"
df_an = merged[merged.category==c]
print(df_an[df_an.delta_fc < 0].gain.sum())
print(df_an[df_an.delta_fc >=0].gain.sum())

480.84500000000196
0.0


In [94]:
df_an[df_an.delta_fc >=0].gain.sum()

np.float64(0.0)

In [95]:
import plotly.express as px

In [96]:
cat = "inflation rate"
df_cat = merged[merged.category==cat]
for p in pairs:
    df_plot = df_cat[df_cat.pair ==  p]
    print(p)
    fig = px.scatter(df_plot, x="gain", y="delta_prev", trendline="ols")
    fig.show()

NZD_USD


USD_PLN


USD_HUF


USD_CNH


USD_CZK


...

USD_TRY


AUD_USD


GBP_USD


c:\Users\otavi\AppData\Local\pypoetry\Cache\virtualenvs\trade-framework-nii59TSx-py3.12\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning:

divide by zero encountered in scalar divide



USD_MXN
